# S2-PR-04 — T2 Logistic Regression and LightGBM

A public-evidence walkthrough. This notebook does not open private parquet or fit a model.
Results are validation-selected baselines, not the final neural R1 or a QR result.

## 1. Locate the repository and the completed batch
Run the documented CLI stages first. Select a different batch only when its full evidence exists.

In [ ]:
from pathlib import Path
import json

ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents) if (p / "pyproject.toml").is_file()), None)
if ROOT is None:
    raise RuntimeError("Open this notebook from within the repository")
BATCH = "batch-001"
PUBLIC = ROOT / "docs/evidence/s2-pr-04-05"
summary_path = PUBLIC / BATCH / "classical_summary.v1.json"
if not summary_path.is_file():
    raise FileNotFoundError("Complete the real baseline stage before viewing results")
summary = json.loads(summary_path.read_text(encoding="utf-8"))
plan = json.loads((ROOT / "config/baselines/s2_pr_04_05.v1.json").read_text(encoding="utf-8"))
print("Status:", summary["status"], "| Batch:", BATCH, "| Task:", summary["task"])

## 2. Data lineage
Frozen TaskExamples supply membership, labels and masks. Canonical raw events supply actual past context. Every anchor binds to the producer’s split-global order; labels are not rebuilt.

In [ ]:
views = json.loads((PUBLIC / "data_views.v1.json").read_text(encoding="utf-8"))
print("Base C1 users:", views["base_c1_users"])
print("Official example-slice users:", views["example_slice_users"])
print("Raw identity status:", views["raw_identity_status"])
print("Anchor bindings:", views["binding"])
print("TEST consumed by this path:", views["test_consumed"])

## 3. The predeclared model and evaluation rules
The reported search choices were fixed before the measurements. Model selection uses VALIDATION only; TEST stays sealed.

In [ ]:
print(plan["classical_selection"])
print("Calibration:", plan["calibration"])
print("Predictor count:", len(json.loads((ROOT / "config/baselines/t2_feature_manifest.v1.json").read_text())["features"]))

## 4. Candidate results
No private IDs or prediction rows are displayed.

In [ ]:
[{"candidate": r["id"], "parameters": r["parameters"], "selection_value": r["selection_value"], "result_uri": r["result_ref"]["uri"]} for r in summary["trials"]]

## 5. Selected result and repeatability
T2 repeats both selected fits from fresh state. T1 verifies the persisted index on the explicitly reported fixed query count; this is not a claim of two full evaluation runs.

In [ ]:
[{"model": w["selected"]["model"], "selected": w["selected"]["id"], "max_abs_probability_diff": w["max_abs_probability_diff"]} for w in summary["winners"]]

## 6. Detailed metrics and true TRAIN-history strata
T2 retains pooled AP and the upstream positive-client top-three companion, not invented macro AP. T1 reports category-change macro MRR@20 with micro and overall diagnostics.

In [ ]:
selected = [w["selected"] for w in summary["winners"]]

In [ ]:
for record in selected:
    detail = json.loads((ROOT / record["metrics_ref"]["uri"]).read_text(encoding="utf-8"))
    print(record["id"])
    print(json.dumps(detail, indent=2))

## 7. Integrity and limitations
Private fitted models, indices, user histories and per-decision predictions stay local. These baselines use the official frozen example slice, not the entire C1 population and not TEST. They do not replace #54 final R1 or #55 multi-task FedAvg.

In [ ]:
verification_path = PUBLIC / BATCH / "verification.v1.json"
if verification_path.is_file():
    check = json.loads(verification_path.read_text(encoding="utf-8"))
    print({k: check[k] for k in ("status", "result_count", "checked_references")})
else:
    print("Final cross-task verification has not been recorded yet")
print(summary["limitations"])